## 05 — Synapse SQL Scripts (Reference & Validation)

This notebook contains the complete Synapse SQL scripts to be run in **Synapse Studio → Develop → SQL Script**.
Run each block separately on the **Built-in** serverless pool with database set to `retail_gold`.

**Databricks workspace :** `https://<YOUR_DATABRICKS_WORKSPACE_URL>`  
**Synapse workspace   :** `synws-retail-analytics`  
**ADLS account        :** `stretaildatalake122`

### Block 1 — Create Database
```sql
-- Run on: Built-in pool, USE DATABASE: master
CREATE DATABASE retail_gold;
```

### Block 2 — Master Key
```sql
-- Switch USE DATABASE to retail_gold first
CREATE MASTER KEY ENCRYPTION BY PASSWORD = '<YOUR_SYNAPSE_MASTER_KEY_PASSWORD>';
```

### Block 3 — Managed Identity Credential
```sql
CREATE DATABASE SCOPED CREDENTIAL adls_credential
WITH IDENTITY = 'Managed Identity';
```

### Block 4 — External Data Sources
```sql
CREATE EXTERNAL DATA SOURCE gold_adls
WITH (
    LOCATION   = 'abfss://gold@stretaildatalake122.dfs.core.windows.net',
    CREDENTIAL = adls_credential
);

CREATE EXTERNAL DATA SOURCE silver_adls
WITH (
    LOCATION   = 'abfss://silver@stretaildatalake122.dfs.core.windows.net',
    CREDENTIAL = adls_credential
);
```

### Block 5 — File Formats
```sql
CREATE EXTERNAL FILE FORMAT delta_format   WITH (FORMAT_TYPE = DELTA);
CREATE EXTERNAL FILE FORMAT parquet_format WITH (FORMAT_TYPE = PARQUET);
```

### Block 6 — Schema
```sql
CREATE SCHEMA gold;
```

### Block 7 — Gold Views
```sql
CREATE VIEW gold.vw_sales_by_product AS
SELECT * FROM OPENROWSET(
    BULK 'sales_summary/by_product/', DATA_SOURCE = 'gold_adls', FORMAT = 'DELTA'
) AS r;

CREATE VIEW gold.vw_sales_by_city AS
SELECT * FROM OPENROWSET(
    BULK 'sales_summary/by_city/', DATA_SOURCE = 'gold_adls', FORMAT = 'DELTA'
) AS r;

CREATE VIEW gold.vw_sales_by_payment AS
SELECT * FROM OPENROWSET(
    BULK 'sales_summary/by_payment/', DATA_SOURCE = 'gold_adls', FORMAT = 'DELTA'
) AS r;
```

### Block 8 — Date-Partitioned Query (Silver)
```sql
-- April 2026 only — partition pruning on ingested_at
SELECT product, city,
       COUNT(*)          AS total_orders,
       SUM(total_amount) AS total_revenue,
       AVG(total_amount) AS avg_order_value
FROM OPENROWSET(
    BULK 'sales_clean/', DATA_SOURCE = 'silver_adls', FORMAT = 'DELTA'
) AS r
WHERE ingested_at >= '2026-04-01'
  AND ingested_at <  '2026-05-01'
GROUP BY product, city
ORDER BY total_revenue DESC;
```

### Block 9 — Hourly Trend
```sql
SELECT LEFT(CAST(ingested_at AS VARCHAR(50)), 13) AS hour_bucket,
       COUNT(*)          AS orders_in_hour,
       SUM(total_amount) AS revenue_in_hour
FROM OPENROWSET(
    BULK 'sales_clean/', DATA_SOURCE = 'silver_adls', FORMAT = 'DELTA'
) AS r
GROUP BY LEFT(CAST(ingested_at AS VARCHAR(50)), 13)
ORDER BY hour_bucket;
```

### Block 10 — High-Value Orders
```sql
SELECT city, product,
       COUNT(*)          AS high_value_orders,
       SUM(total_amount) AS high_value_revenue
FROM OPENROWSET(
    BULK 'sales_clean/', DATA_SOURCE = 'silver_adls', FORMAT = 'DELTA'
) AS r
WHERE is_high_value = 'true'
GROUP BY city, product
ORDER BY high_value_revenue DESC;
```

### Block 11 — CETAS (Materialise April 2026 Summary)
```sql
-- NOTE: CETAS cannot write Delta in Serverless pool — Parquet is used instead.
CREATE EXTERNAL TABLE gold.sales_apr2026_summary
WITH (
    LOCATION    = 'sales_summary/apr2026_loaded/',
    DATA_SOURCE = gold_adls,
    FILE_FORMAT = parquet_format
)
AS
SELECT product, city,
       COUNT(*)          AS total_orders,
       SUM(total_amount) AS total_revenue,
       SUM(quantity)     AS units_sold,
       MIN(ingested_at)  AS first_event,
       MAX(ingested_at)  AS last_event
FROM OPENROWSET(
    BULK 'sales_clean/', DATA_SOURCE = 'silver_adls', FORMAT = 'DELTA'
) AS r
WHERE ingested_at >= '2026-04-01'
GROUP BY product, city;

-- Verify:
SELECT * FROM gold.sales_apr2026_summary ORDER BY total_revenue DESC;
```

### Block 12 — Validate Managed Identity
```sql
SELECT name AS credential_name, credential_identity AS identity_type
FROM sys.database_scoped_credentials
WHERE name = 'adls_credential';
-- Expected: identity_type = 'Managed Identity'
```